# Notebook 08 — Clasificación Factorial de Imágenes

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña  
**Fase:** 1 del pipeline de validación — Clasificación factorial

---

## ¿Qué hace este notebook?

Toma las imágenes del conjunto de validación y las clasifica según el diseño factorial 2×2×2:

- **Factor 1 — Tipo de tarea:** KITTI (discriminación real) vs. Ilusiones visuales
- **Factor 2 — Nivel de disparidad binocular:** Alta vs. Baja
- **Factor 3 — Presencia de distractores:** Con vs. Sin

Resultado: archivo `factorial_labels.csv` con cada imagen etiquetada. Es la base de todo el análisis posterior.

**Criterios técnicos:**
- Disparidad (KITTI): mapas ground truth `disp_occ_0`. Umbral = mediana de la distribución.
- Distractores: densidad de bordes Sobel. Umbral = mediana de la distribución.
- Para ilusiones: densidad de bordes como proxy de fuerza ilusoria (reemplaza disparidad).

---

## Celda 1 — Montar Google Drive y verificar rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR       = '/content/drive/MyDrive/cognitive-depth-model'

KITTI_DISP_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'kitti',
                              'data_scene_flow', 'training', 'disp_occ_0')
KITTI_LEFT_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'kitti',
                              'data_scene_flow', 'training', 'image_2')
ILLUSION_DIR   = os.path.join(BASE_DIR, 'data', 'raw', 'illusions', 'images')
SPLITS_DIR     = os.path.join(BASE_DIR, 'data', 'splits')
RESULTS_DIR    = os.path.join(BASE_DIR, 'results', 'factorial')
os.makedirs(RESULTS_DIR, exist_ok=True)

rutas = {
    'KITTI disparidad (disp_occ_0)': KITTI_DISP_DIR,
    'KITTI imágenes (image_2)':      KITTI_LEFT_DIR,
    'Ilusiones (images/)':           ILLUSION_DIR,
    'Splits':                        SPLITS_DIR,
    'Resultados':                    RESULTS_DIR,
}

print('Verificación de rutas:')
print('-' * 55)
todo_ok = True
for nombre, ruta in rutas.items():
    existe = os.path.exists(ruta)
    if not existe:
        todo_ok = False
    print(f'  {"✓ Existe" if existe else "✗ NO ENCONTRADA"}  {nombre}')

print()
if todo_ok:
    print('✓ Todas las rutas OK. Puedes continuar con la Celda 2.')
else:
    print('✗ Alguna ruta falla. Detente y reporta el error antes de continuar.')

## Celda 2 — Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print('Librerías importadas correctamente.')
print(f'  NumPy:  {np.__version__}')
print(f'  OpenCV: {cv2.__version__}')
print(f'  Pandas: {pd.__version__}')

## Celda 3 — Cargar e inspeccionar los splits JSON

In [ ]:
# Cargar splits de KITTI
with open(os.path.join(SPLITS_DIR, 'kitti_splits.json'), 'r') as f:
    kitti_splits = json.load(f)

print('=== kitti_splits.json ===')
for clave, valor in kitti_splits.items():
    if isinstance(valor, list):
        print(f'  "{clave}": {len(valor)} elementos  |  primeros 3: {valor[:3]}')
    else:
        print(f'  "{clave}": {valor}')

print()

# Cargar splits de ilusiones
with open(os.path.join(SPLITS_DIR, 'illusion_splits.json'), 'r') as f:
    illusion_splits = json.load(f)

print('=== illusion_splits.json ===')
for clave, valor in illusion_splits.items():
    if isinstance(valor, list):
        print(f'  "{clave}": {len(valor)} elementos  |  primer elemento: {str(valor[0])[:80]}')
    else:
        print(f'  "{clave}": {valor}')

print()
print('--- Verificar formato de IDs vs archivos en disco ---')

# Detectar clave de validación en KITTI
clave_val_kitti = next((c for c in ['validation','val','test'] if c in kitti_splits), None)
kitti_val_raw   = kitti_splits.get(clave_val_kitti, [])
print(f'KITTI  → clave "{clave_val_kitti}": {len(kitti_val_raw)} IDs')

# Detectar clave de validación en ilusiones
clave_val_ill   = next((c for c in ['validation','val','test'] if c in illusion_splits), None)
illusion_val_raw = illusion_splits.get(clave_val_ill, [])
print(f'Ilusiones → clave "{clave_val_ill}": {len(illusion_val_raw)} IDs')

# Verificar que el primer ID de KITTI encuentra archivo real
if kitti_val_raw:
    primer = kitti_val_raw[0]
    for sufijo in ['', '_10', '_10']:
        nombre = f'{str(primer).zfill(6)}{sufijo}.png' if sufijo else f'{primer}.png'
        existe = (Path(KITTI_DISP_DIR) / nombre).exists()
        print(f'  KITTI busca "{nombre}" en disp_occ_0 → {"✓" if existe else "✗"}')
        if existe:
            break

# Verificar que el primer ID de ilusiones encuentra archivo real
if illusion_val_raw:
    primer_ill = illusion_val_raw[0]
    for ext in ['.png', '.jpg']:
        nombre_ill = f'{primer_ill}{ext}'
        existe_ill = (Path(ILLUSION_DIR) / nombre_ill).exists()
        print(f'  Ilusión busca "{str(nombre_ill)[:60]}" → {"✓" if existe_ill else "✗"}')
        if existe_ill:
            break

## Celda 4 — Preparar listados definitivos de IDs de validación

In [ ]:
# KITTI: usar split de validación; si no existe, tomar 100 desde disco
if kitti_val_raw:
    kitti_val_ids = kitti_val_raw
else:
    print('Sin clave de validación en KITTI JSON. Usando archivos del disco.')
    kitti_val_ids = [f.stem for f in sorted(Path(KITTI_DISP_DIR).glob('*_10.png'))[:100]]

# Ilusiones: usar split de validación; si no existe, tomar 100 desde disco
if illusion_val_raw:
    illusion_val_ids = illusion_val_raw
else:
    print('Sin clave de validación en Ilusiones JSON. Usando archivos del disco.')
    files_ill = sorted(list(Path(ILLUSION_DIR).glob('*.png')) +
                       list(Path(ILLUSION_DIR).glob('*.jpg')))
    illusion_val_ids = [f.stem for f in files_ill[:100]]

print(f'IDs de validación KITTI:     {len(kitti_val_ids)}')
print(f'IDs de validación Ilusiones: {len(illusion_val_ids)}')
print(f'Total:                       {len(kitti_val_ids) + len(illusion_val_ids)}')

## Celda 5 — Definir funciones de clasificación

In [ ]:
def resolver_ruta_kitti(img_id, carpeta):
    """Resuelve la ruta real de un archivo KITTI con varios formatos de ID posibles."""
    carpeta = Path(carpeta)
    id_str = str(img_id)
    candidatos = [
        carpeta / f'{id_str}.png',
        carpeta / f'{id_str}_10.png',
        carpeta / f'{id_str.zfill(6)}_10.png',
        carpeta / f'{id_str.zfill(6)}.png',
    ]
    for c in candidatos:
        if c.exists():
            return c
    return None


def calcular_disparidad_media(ruta_png):
    """
    Lee un mapa de disparidad KITTI (PNG 16-bit).
    Conversión: valor_real = pixel_PNG / 256.0
    Píxeles con valor 0 son inválidos (sin dato LiDAR) y se excluyen.
    Retorna la media de píxeles válidos, o np.nan si no hay datos.
    """
    mapa = cv2.imread(str(ruta_png), cv2.IMREAD_UNCHANGED)
    if mapa is None:
        return np.nan
    mapa_real = mapa.astype(np.float32) / 256.0
    validos = mapa_real[mapa_real > 0]
    return float(np.mean(validos)) if len(validos) > 0 else np.nan


def calcular_densidad_bordes(ruta_imagen):
    """
    Calcula la densidad de bordes de una imagen como medida de
    complejidad visual (proxy de distractores).
    Usa gradientes Sobel. Resultado normalizado entre 0 y 1.
    Mayor densidad = más elementos visuales = más distractores.
    """
    img = cv2.imread(str(ruta_imagen), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan
    img = cv2.resize(img, (640, 192))
    img_suave = cv2.GaussianBlur(img, (3, 3), 0)
    gx = cv2.Sobel(img_suave, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_suave, cv2.CV_64F, 0, 1, ksize=3)
    magnitud = np.sqrt(gx**2 + gy**2)
    return float(np.mean(magnitud) / 1442.0)


print('Funciones definidas correctamente.')
print()
print('Lógica de clasificación:')
print('  Disparidad KITTI → mediana del mapa disp_occ_0 (umbral relativo)')
print('  Distractores     → densidad Sobel (umbral relativo = mediana global)')
print('  Disparidad ilusiones → densidad Sobel interna (proxy de fuerza ilusoria)')

## Celda 6 — Procesar imágenes KITTI

In [ ]:
print('Procesando imágenes KITTI...')
print('Tiempo estimado: 1-2 minutos.\n')

registros_kitti   = []
no_encontradas_k  = []

for img_id in tqdm(kitti_val_ids, desc='KITTI'):
    ruta_disp = resolver_ruta_kitti(img_id, KITTI_DISP_DIR)
    ruta_img  = resolver_ruta_kitti(img_id, KITTI_LEFT_DIR)

    if ruta_disp is None and ruta_img is None:
        no_encontradas_k.append(img_id)

    registros_kitti.append({
        'imagen_id':        str(img_id),
        'dataset':          'KITTI',
        'tipo_tarea':       'discriminacion_profundidad',
        'tipo_ilusion':     '',
        'ruta_img_izq':     str(ruta_img)  if ruta_img  else '',
        'ruta_disp':        str(ruta_disp) if ruta_disp else '',
        'disparidad_media': calcular_disparidad_media(ruta_disp) if ruta_disp else np.nan,
        'densidad_bordes':  calcular_densidad_bordes(ruta_img)   if ruta_img  else np.nan,
    })

df_kitti = pd.DataFrame(registros_kitti)

print(f'\nProcesadas:                  {len(df_kitti)}')
print(f'Archivos no encontrados:     {len(no_encontradas_k)}')
print(f'Disparidades válidas:        {df_kitti["disparidad_media"].notna().sum()}')
print(f'Densidades de borde válidas: {df_kitti["densidad_bordes"].notna().sum()}')
print()
print('Estadísticas de disparidad media (KITTI):')
print(df_kitti['disparidad_media'].describe().round(4))

## Celda 7 — Procesar imágenes de Ilusiones Visuales

In [ ]:
print('Procesando imágenes de Ilusiones Visuales...')
print('Tiempo estimado: 1-2 minutos.\n')

# Índice de todos los archivos disponibles en images/
indice_ill = {}
for ext in ['*.png', '*.jpg', '*.jpeg']:
    for ruta in Path(ILLUSION_DIR).glob(ext):
        indice_ill[ruta.stem] = ruta
print(f'Archivos disponibles en images/: {len(indice_ill)}')

registros_ill = []
no_encontradas_i = []

for img_id in tqdm(illusion_val_ids, desc='Ilusiones'):
    id_str   = str(img_id)
    ruta_img = indice_ill.get(id_str, None)
    if ruta_img is None:
        # Búsqueda parcial por si el ID viene sin extensión o con variación
        for stem, ruta in indice_ill.items():
            if id_str in stem or stem in id_str:
                ruta_img = ruta
                break
    if ruta_img is None:
        no_encontradas_i.append(img_id)

    # Tipo de ilusión desde el nombre del archivo
    nombre_lower = id_str.lower()
    if 'mirror'     in nombre_lower: tipo_il = 'mirror'
    elif 'holograph' in nombre_lower: tipo_il = 'holography'
    elif 'replay'    in nombre_lower: tipo_il = 'replay'
    elif 'picture'   in nombre_lower: tipo_il = 'picture'
    elif 'inpaint'   in nombre_lower: tipo_il = 'inpainting'
    else:                             tipo_il = 'otro'

    registros_ill.append({
        'imagen_id':        id_str,
        'dataset':          '3D_Illusion',
        'tipo_tarea':       'ilusion_visual',
        'tipo_ilusion':     tipo_il,
        'ruta_img_izq':     str(ruta_img) if ruta_img else '',
        'ruta_disp':        '',
        'disparidad_media': np.nan,
        'densidad_bordes':  calcular_densidad_bordes(ruta_img) if ruta_img else np.nan,
    })

df_illusion = pd.DataFrame(registros_ill)

print(f'\nProcesadas:                  {len(df_illusion)}')
print(f'Archivos no encontrados:     {len(no_encontradas_i)}')
print(f'Densidades de borde válidas: {df_illusion["densidad_bordes"].notna().sum()}')
print()
print('Tipos de ilusión detectados:')
print(df_illusion['tipo_ilusion'].value_counts().to_string())

## Celda 8 — Calcular umbrales y asignar etiquetas factoriales

In [ ]:
df_total = pd.concat([df_kitti, df_illusion], ignore_index=True)
print(f'Total combinado: {len(df_total)}  (KITTI: {len(df_kitti)} | Ilusiones: {len(df_illusion)})')
print()

# Umbrales por mediana
UMBRAL_DISPARIDAD  = float(df_total.loc[df_total['dataset']=='KITTI',
                                         'disparidad_media'].dropna().median())
UMBRAL_BORDES      = float(df_total['densidad_bordes'].dropna().median())
UMBRAL_BORDES_ILL  = float(df_total.loc[df_total['dataset']=='3D_Illusion',
                                          'densidad_bordes'].dropna().median())

print(f'Umbral disparidad KITTI:       {UMBRAL_DISPARIDAD:.4f} px')
print(f'Umbral distractores (global):  {UMBRAL_BORDES:.6f}')
print(f'Umbral disp. ilusiones (int.): {UMBRAL_BORDES_ILL:.6f}')
print()

def etiquetar_disparidad(row):
    if row['dataset'] == 'KITTI':
        if pd.isna(row['disparidad_media']): return 'sin_dato'
        return 'alta' if row['disparidad_media'] >= UMBRAL_DISPARIDAD else 'baja'
    else:
        if pd.isna(row['densidad_bordes']): return 'sin_dato'
        return 'alta' if row['densidad_bordes'] >= UMBRAL_BORDES_ILL else 'baja'

def etiquetar_distractores(row):
    if pd.isna(row['densidad_bordes']): return 'sin_dato'
    return 'con_distractores' if row['densidad_bordes'] >= UMBRAL_BORDES else 'sin_distractores'

df_total['nivel_disparidad']       = df_total.apply(etiquetar_disparidad,   axis=1)
df_total['presencia_distractores'] = df_total.apply(etiquetar_distractores, axis=1)

print('Distribución factorial resultante (N imágenes por celda):')
print('=' * 65)
tabla = df_total.groupby(
    ['tipo_tarea', 'nivel_disparidad', 'presencia_distractores']
).size().reset_index(name='N')
print(tabla.to_string(index=False))
print()
print('(Ideal: N = 25 en cada celda para un diseño perfectamente balanceado)')

## Celda 9 — Visualizaciones de las distribuciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Clasificación Factorial — Distribuciones de Disparidad y Complejidad Visual',
             fontsize=13, fontweight='bold')

# 1. Distribución disparidad KITTI
ax = axes[0, 0]
kd = df_total.loc[df_total['dataset']=='KITTI', 'disparidad_media'].dropna()
ax.hist(kd, bins=25, color='#90CAF9', edgecolor='white', alpha=0.85)
ax.axvline(UMBRAL_DISPARIDAD, color='#D32F2F', lw=2, ls='--',
           label=f'Mediana = {UMBRAL_DISPARIDAD:.2f} px')
ax.set_xlabel('Disparidad media (píxeles)')
ax.set_ylabel('N escenas')
ax.set_title('Distribución de disparidad — KITTI')
ax.legend(fontsize=9)
ax.text(0.05, 0.88, f'Alta: {(kd>=UMBRAL_DISPARIDAD).sum()}  |  Baja: {(kd<UMBRAL_DISPARIDAD).sum()}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# 2. Distribución densidad de bordes
ax = axes[0, 1]
for ds, color, lbl in [('KITTI','#2196F3','KITTI'),('3D_Illusion','#9C27B0','Ilusiones')]:
    bd = df_total.loc[df_total['dataset']==ds, 'densidad_bordes'].dropna()
    ax.hist(bd, bins=20, alpha=0.6, color=color, label=lbl, edgecolor='white')
ax.axvline(UMBRAL_BORDES, color='#D32F2F', lw=2, ls='--',
           label=f'Mediana = {UMBRAL_BORDES:.4f}')
ax.set_xlabel('Densidad de bordes normalizada')
ax.set_ylabel('N imágenes')
ax.set_title('Complejidad visual (proxy de distractores)')
ax.legend(fontsize=9)

# 3. Balance factorial KITTI
ax = axes[1, 0]
bal_k = df_total[df_total['dataset']=='KITTI'].groupby(
    ['nivel_disparidad','presencia_distractores']).size()
etiq_k = [f'{d}\n{p.replace("_"," ")}' for d,p in bal_k.index]
cols_k = ['#42A5F5','#1565C0','#FFA726','#E65100']
bars_k = ax.bar(range(len(bal_k)), bal_k.values, color=cols_k[:len(bal_k)], edgecolor='white')
ax.set_xticks(range(len(bal_k))); ax.set_xticklabels(etiq_k, fontsize=9)
ax.set_ylabel('N imágenes'); ax.set_title('Balance factorial — KITTI')
ax.axhline(25, color='red', ls=':', lw=1.5, label='Ideal = 25'); ax.legend(fontsize=8)
for b, v in zip(bars_k, bal_k.values):
    ax.text(b.get_x()+b.get_width()/2, v+0.3, str(v), ha='center', fontsize=11, fontweight='bold')

# 4. Balance factorial Ilusiones
ax = axes[1, 1]
bal_i = df_total[df_total['dataset']=='3D_Illusion'].groupby(
    ['nivel_disparidad','presencia_distractores']).size()
etiq_i = [f'{d}\n{p.replace("_"," ")}' for d,p in bal_i.index]
cols_i = ['#AB47BC','#6A1B9A','#EC407A','#880E4F']
bars_i = ax.bar(range(len(bal_i)), bal_i.values, color=cols_i[:len(bal_i)], edgecolor='white')
ax.set_xticks(range(len(bal_i))); ax.set_xticklabels(etiq_i, fontsize=9)
ax.set_ylabel('N imágenes'); ax.set_title('Balance factorial — Ilusiones Visuales')
ax.axhline(25, color='red', ls=':', lw=1.5, label='Ideal = 25'); ax.legend(fontsize=8)
for b, v in zip(bars_i, bal_i.values):
    ax.text(b.get_x()+b.get_width()/2, v+0.3, str(v), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_fig = os.path.join(RESULTS_DIR, 'factorial_distributions.png')
plt.savefig(ruta_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en: {ruta_fig}')

## Celda 10 — Guardar CSV maestro

In [ ]:
columnas = [
    'imagen_id', 'dataset', 'tipo_tarea', 'tipo_ilusion',
    'nivel_disparidad', 'presencia_distractores',
    'disparidad_media', 'densidad_bordes',
    'ruta_img_izq', 'ruta_disp'
]
df_final = df_total[columnas].copy()

ruta_csv_results = os.path.join(RESULTS_DIR, 'factorial_labels.csv')
ruta_csv_splits  = os.path.join(SPLITS_DIR,  'factorial_labels.csv')
df_final.to_csv(ruta_csv_results, index=False)
df_final.to_csv(ruta_csv_splits,  index=False)

print('CSV maestro guardado exitosamente.')
print(f'  → {ruta_csv_results}')
print(f'  → {ruta_csv_splits}')
print(f'\nTotal registros: {len(df_final)}')
print()
print('Vista previa:')
print(df_final[['imagen_id','dataset','nivel_disparidad',
                'presencia_distractores','disparidad_media',
                'densidad_bordes']].head(8).to_string(index=False))

## Celda 11 — Resumen final y verificación de balance

In [ ]:
print('=' * 65)
print('RESUMEN FINAL — CLASIFICACIÓN FACTORIAL')
print('=' * 65)
print(f'Total imágenes clasificadas: {len(df_final)}')
print(f'  KITTI (discriminación):    {(df_final["dataset"]=="KITTI").sum()}')
print(f'  Ilusiones visuales:        {(df_final["dataset"]=="3D_Illusion").sum()}')
print()
print(f'Umbral disparidad:  {UMBRAL_DISPARIDAD:.4f} px  (mediana KITTI)')
print(f'Umbral distractores:{UMBRAL_BORDES:.6f}  (mediana global Sobel norm.)')
print()
print('Tabla factorial 2×2×2 completa:')
print('-' * 65)
tabla_final = df_final.groupby(
    ['tipo_tarea', 'nivel_disparidad', 'presencia_distractores']
).size().reset_index(name='N')
print(tabla_final.to_string(index=False))
print()

ns = tabla_final['N']
if ns.min() >= 20 and ns.max() <= 30:
    print('✓ Diseño balanceado (todas las celdas entre 20 y 30 imágenes).')
else:
    print('⚠ Hay celdas con desequilibrio.')
    print(f'  Mínimo: {ns.min()} | Máximo: {ns.max()} | Ideal: 25')
    print('  Se corregirá en el Notebook 09 al seleccionar el subconjunto balanceado.')

print()
print('Archivos generados:')
print(f'  1. results/factorial/factorial_labels.csv')
print(f'  2. data/splits/factorial_labels.csv')
print(f'  3. results/factorial/factorial_distributions.png')
print()
print('Próximo paso → Notebook 09: Definición de pares A/B y etiquetas ground truth.')